# 04 — Train and evaluate the smoke-aware residual LSTM

Select training duration on validation data, evaluate once on the untouched August 2026 smoke period, and only then build the production artifact from all available data.

In [ ]:
import gc
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'common').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling'))

import joblib
import numpy as np
import torch
import torch.nn.functional as F
import wandb
from torch.utils.data import DataLoader

from dataset import SequenceDataset, smoke_sampler
from model import LSTMRegressor
from splits import apply_scaler, fit_feature_scaler
from windowing import FEATURE_COLS, HORIZON, SEQ_LEN, STATION_FEATURES

torch.manual_seed(0)
np.random.seed(0)
# CPU is deliberate: MPS produced different smoke metrics from the same seeds.
# This dataset trains quickly on CPU and matches the deployment environment.
device = torch.device('cpu')

ARTIFACT_DIR = ROOT / 'modeling' / 'artifacts' / 'v3'
windows = np.load(ARTIFACT_DIR / 'windows.npz')

X_raw = windows['X']
y = windows['y']
current = windows['current']
train_index = windows['train']
validation_index = windows['validation']
test_index = windows['test']
stations = windows['station']

config = {
    'hidden': 64,
    'layers': 1,
    'dropout': 0.0,
    'learning_rate': 1e-3,
    'batch_size': 64,
    'maximum_epochs': 80,
    'patience': 12,
    'sequence_length': SEQ_LEN,
    'horizon': HORIZON,
    'residual': True,
}

run = wandb.init(
    project='airquality-pm25',
    name='residual-smoke-aware-v3-evaluation',
    config=config,
)

print('device:', device)
print('windows:', X_raw.shape, '| features:', len(FEATURE_COLS))

In [ ]:
def new_model():
    return LSTMRegressor(
        n_features=len(FEATURE_COLS),
        hidden=config['hidden'],
        layers=config['layers'],
        dropout=config['dropout'],
        residual=True,
    ).to(device)


def make_loader(X, target, current_pm25, training):
    dataset = SequenceDataset(X, target, current_pm25)
    if training:
        return DataLoader(
            dataset,
            batch_size=config['batch_size'],
            sampler=smoke_sampler(target, current_pm25),
        )
    return DataLoader(dataset, batch_size=512, shuffle=False)


def smoke_loss(prediction, target):
    weights = torch.ones_like(target)
    weights = torch.where(target > 12.0, 1.5, weights)
    weights = torch.where(target > 35.4, 3.0, weights)
    weights = torch.where(target > 55.4, 5.0, weights)
    underprediction = (target > 35.4) & (prediction < target)
    weights = torch.where(underprediction, weights * 1.25, weights)
    raw_loss = F.huber_loss(
        prediction, target, reduction='none', delta=10.0
    )
    return (raw_loss * weights).sum() / weights.sum()


def train_epoch(model, loader, optimizer):
    model.train()
    losses = []
    for X_batch, y_batch, current_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        current_batch = current_batch.to(device)
        optimizer.zero_grad()
        prediction = model(X_batch, current_batch)
        loss = smoke_loss(prediction, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def predict(model, loader):
    model.eval()
    predictions, targets, current_values = [], [], []
    with torch.no_grad():
        for X_batch, y_batch, current_batch in loader:
            prediction = model(
                X_batch.to(device), current_batch.to(device)
            )
            predictions.append(prediction.cpu().numpy().ravel())
            targets.append(y_batch.numpy().ravel())
            current_values.append(current_batch.numpy().ravel())
    return (
        np.concatenate(predictions),
        np.concatenate(targets),
        np.concatenate(current_values),
    )


def forecast_metrics(target, prediction, current_pm25):
    error = prediction - target
    high = target > 35.4
    onset = high & (current_pm25 <= 12.0)
    normal = ~high
    return {
        'overall_mae': float(np.abs(error).mean()),
        'rmse': float(np.sqrt(np.square(error).mean())),
        'high_n': int(high.sum()),
        'high_mae': float(np.abs(error[high]).mean()) if high.any() else float('nan'),
        'onset_n': int(onset.sum()),
        'onset_mae': float(np.abs(error[onset]).mean()) if onset.any() else float('nan'),
        'severe_underprediction_rate': (
            float(((target[high] - prediction[high]) > 20.0).mean())
            if high.any() else float('nan')
        ),
        'exceedance_recall': (
            float((prediction[high] > 35.4).mean())
            if high.any() else float('nan')
        ),
        'false_alarm_rate': (
            float((prediction[normal] > 35.4).mean())
            if normal.any() else float('nan')
        ),
        'maximum_prediction': float(prediction.max()),
    }


def checkpoint_score(metrics):
    return (
        metrics['overall_mae']
        + 0.25 * metrics['high_mae']
        + 5.0 * metrics['severe_underprediction_rate']
        + 2.0 * metrics['false_alarm_rate']
    )


def train_fixed_epochs(X, target, current_pm25, epochs):
    torch.manual_seed(0)
    model = new_model()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config['learning_rate']
    )
    loader = make_loader(X, target, current_pm25, training=True)
    for epoch in range(epochs):
        loss = train_epoch(model, loader, optimizer)
        print(f'fixed epoch={epoch + 1:02d} loss={loss:.3f}')
    return model

In [ ]:
candidate_scaler = fit_feature_scaler(X_raw[train_index])
X_train = apply_scaler(candidate_scaler, X_raw[train_index])
X_validation = apply_scaler(candidate_scaler, X_raw[validation_index])

train_loader = make_loader(
    X_train, y[train_index], current[train_index], training=True
)
validation_loader = make_loader(
    X_validation, y[validation_index], current[validation_index], training=False
)
candidate_model = new_model()
optimizer = torch.optim.Adam(
    candidate_model.parameters(), lr=config['learning_rate']
)

best_score = float('inf')
best_epochs = 1
stale_epochs = 0

for epoch in range(config['maximum_epochs']):
    train_loss = train_epoch(candidate_model, train_loader, optimizer)
    prediction, target, current_value = predict(
        candidate_model, validation_loader
    )
    validation_metrics = forecast_metrics(target, prediction, current_value)
    score = checkpoint_score(validation_metrics)
    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'checkpoint_score': score,
        **{
            f'validation/{key}': value
            for key, value in validation_metrics.items()
        },
    })
    print(
        f"epoch={epoch + 1:02d} "
        f"overall={validation_metrics['overall_mae']:.2f} "
        f"high={validation_metrics['high_mae']:.2f} "
        f"recall={validation_metrics['exceedance_recall']:.1%} "
        f"false_alarm={validation_metrics['false_alarm_rate']:.1%}"
    )
    if score < best_score:
        best_score = score
        best_epochs = epoch + 1
        stale_epochs = 0
    else:
        stale_epochs += 1
    if stale_epochs >= config['patience']:
        break

print('selected epochs:', best_epochs)

del X_train, X_validation, train_loader, validation_loader
del candidate_model, candidate_scaler
gc.collect()

In [ ]:
development_index = np.concatenate([train_index, validation_index])
evaluation_scaler = fit_feature_scaler(X_raw[development_index])
X_development = apply_scaler(evaluation_scaler, X_raw[development_index])
X_test = apply_scaler(evaluation_scaler, X_raw[test_index])

evaluation_model = train_fixed_epochs(
    X_development,
    y[development_index],
    current[development_index],
    epochs=best_epochs,
)
test_loader = make_loader(
    X_test, y[test_index], current[test_index], training=False
)
prediction, target, current_value = predict(evaluation_model, test_loader)
v3_metrics = forecast_metrics(target, prediction, current_value)
persistence_metrics = forecast_metrics(target, current_value, current_value)

print('\nV3 August smoke test')
print(json.dumps(v3_metrics, indent=2))
print('\n24-hour persistence')
print(json.dumps(persistence_metrics, indent=2))

for station_id in np.unique(stations[test_index]):
    mask = stations[test_index] == station_id
    station_metrics = forecast_metrics(
        target[mask], prediction[mask], current_value[mask]
    )
    print(f'\n{station_id}')
    print(json.dumps(station_metrics, indent=2))

wandb.log({
    **{f'smoke_test/v3_{key}': value for key, value in v3_metrics.items()},
    **{
        f'smoke_test/persistence_{key}': value
        for key, value in persistence_metrics.items()
    },
})
wandb.finish()

del X_development, X_test, evaluation_model, evaluation_scaler
gc.collect()

In [ ]:
# The artifact cannot be published unless it materially improves smoke behavior.
ACCEPTED_FOR_DEPLOYMENT = (
    v3_metrics['overall_mae'] <= persistence_metrics['overall_mae']
    and v3_metrics['high_mae'] < 0.90 * persistence_metrics['high_mae']
    and v3_metrics['severe_underprediction_rate'] <= 0.50
    and v3_metrics['exceedance_recall'] >= persistence_metrics['exceedance_recall']
    and v3_metrics['false_alarm_rate'] <= 0.10
    and v3_metrics['maximum_prediction'] > 55.4
)
print('accepted for deployment:', ACCEPTED_FOR_DEPLOYMENT)

PUBLISH_V3 = False

production_run = wandb.init(
    project='airquality-pm25',
    name='residual-smoke-aware-v3-production',
    job_type='production-training',
    config={**config, 'selected_epochs': best_epochs},
)
production_scaler = fit_feature_scaler(X_raw)
X_all = apply_scaler(production_scaler, X_raw)
production_model = train_fixed_epochs(X_all, y, current, epochs=best_epochs)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / 'model.pt'
preprocess_path = ARTIFACT_DIR / 'preprocess.joblib'
config_path = ARTIFACT_DIR / 'model_config.json'

torch.save(production_model.state_dict(), model_path)
joblib.dump({
    'scaler': production_scaler,
    'feature_cols': FEATURE_COLS,
    'seq_len': SEQ_LEN,
    'horizon': HORIZON,
    'station_features': STATION_FEATURES,
    'residual': True,
}, preprocess_path)
config_path.write_text(json.dumps({
    'n_features': len(FEATURE_COLS),
    'hidden': config['hidden'],
    'layers': config['layers'],
    'dropout': config['dropout'],
    'residual': True,
}))

if PUBLISH_V3 and not ACCEPTED_FOR_DEPLOYMENT:
    raise RuntimeError('v3 failed its smoke-event deployment gates')

if PUBLISH_V3:
    artifact = wandb.Artifact(
        'pm25-lstm',
        type='model',
        metadata={
            'architecture': 'residual-lstm',
            'august_smoke_test': v3_metrics,
            'persistence_test': persistence_metrics,
        },
    )
    artifact.add_file(str(model_path))
    artifact.add_file(str(preprocess_path))
    artifact.add_file(str(config_path))
    production_run.log_artifact(artifact)
    print('Uploaded pm25-lstm; confirm its exact version before deployment.')
else:
    print('Saved local candidate:', ARTIFACT_DIR)
    print('Set PUBLISH_V3=True and rerun this cell to upload it.')

production_run.finish()